# 第二课 · 用 peft + QLoRA 微调 Qwen2.5-1.5B 做外贸术语 QA

**环境**：Kaggle Notebook（GPU T4 x2）或 Google Colab（T4）；Mac 可加载推理但不适合训练
**任务**：用 500 条 Incoterms + HS 编码的 QA 训练一个「外贸小助手」，在 50 条盲测集上打分
**目标**：
1. 掌握 🤗 `peft` 的 `LoraConfig` + `get_peft_model` + `merge_and_unload` 三板斧
2. 理解 **QLoRA**（4bit 量化 + LoRA）为什么能在 16GB T4 上跑 1.5B 模型
3. 走完「加载数据 → SFT 训练 → 合并权重 → 推理对比 → 盲测打分」完整主线

**预期时长**：
- **Kaggle T4 / Colab T4**：训练 10-20 min + 模型下载首次 5-10 min ≈ 总共 20-30 min
- **Mac MPS**：仅做流程验证（`max_steps=2`），真训练去 GPU 平台

> 🦞 **与第一课的关系**：第一课**手写** `LoRALinear` 打通数学原理；这一课改用工业级工具链（`peft` + `trl` + `bitsandbytes`），重点从「怎么实现」转向「怎么用好」。


---

## 1. 环境检测与依赖安装

本 notebook **自动适配 Kaggle (CUDA) / Colab (CUDA) / Mac (MPS) / CPU**：
- **GPU (CUDA)**：走 QLoRA（4bit 量化 + LoRA），`bitsandbytes` 必装 → **真训练**
- **Mac MPS**：1.5B 模型在 MPS 上训练不实际（每 step 数分钟）→ **仅验证流程**
- **CPU**：极慢，建议换 GPU 平台

🔴 **Kaggle 必做**：Settings → Accelerator = GPU T4 x2，Internet = On
🟢 **Colab 必做**：Runtime → Change runtime type → T4 GPU


In [ ]:
import sys, platform, torch

# 自动选设备
if torch.cuda.is_available():
    DEVICE = "cuda"
    MODE = "qlora"     # Kaggle 走 4bit 量化
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    MODE = "fp16_lora" # Mac 降级为 fp16 LoRA
else:
    DEVICE = "cpu"
    MODE = "fp32_lora" # CPU 兜底（极慢）

print(f"Python   : {sys.version.split()[0]}")
print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {DEVICE}")
print(f"Mode     : {MODE}")
if DEVICE == "cuda":
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"Memory   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


In [ ]:
# 依赖安装
# Kaggle 预装了 torch + transformers，我们补装 peft + trl + datasets + evaluate
# bitsandbytes 只在 CUDA 环境装（Mac/CPU 会跳过）
import subprocess, sys

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

common = [
    "peft>=0.11",
    "trl>=0.9",
    "datasets>=2.18",
    "accelerate>=0.27",
    "evaluate>=0.4",
    "sacrebleu",          # BLEU 评测
    "rouge-score",        # ROUGE 评测
    "jieba",              # 中文分词（BLEU/ROUGE 都要）
]
pip_install(common)

if MODE == "qlora":
    pip_install(["bitsandbytes>=0.43"])
    print("✅ 已装 bitsandbytes（Kaggle QLoRA 专用）")
else:
    print(f"⚪ 跳过 bitsandbytes（当前 MODE={MODE}）")

print("✅ 依赖安装完成")


---

## 2. QLoRA 原理一页纸

**QLoRA = 4bit 量化底座 + LoRA 适配器**（Dettmers et al., NeurIPS 2023）

```
┌─────────────────────────────────────────────────────┐
│  原模型 Qwen2.5-1.5B（fp16 ≈ 3GB） → 4bit ≈ 0.9GB   │
│  ↓                                                    │
│  冻结！不训练！（显存只占 ~1GB）                      │
│  ↓                                                    │
│  每个 Linear 外挂 LoRA (A, B)，fp16 训练            │
│  ↓                                                    │
│  前向：h = dequant(W_4bit) @ x + (α/r) B A x         │
│  反向：只对 A, B 求梯度，w_4bit 不动                 │
└─────────────────────────────────────────────────────┘
```

**三项关键技术**（每项都有论文）：

| 技术 | 解决什么 | 实现细节 |
|------|---------|---------|
| **NF4 量化** | fp16 → 4bit 精度损失太大 | 按权重分布设计的「Normal Float 4」，比 INT4 精度高 |
| **双重量化** | 量化常数本身也占显存 | 对 quant constant 再做一轮 8bit 量化 |
| **Paged Optimizer** | 长序列激活显存峰值 | 把 optimizer state 分页，显存爆了自动换到 CPU |

**效果**（原论文）：65B 模型在 1 张 48GB GPU 上做 SFT，性能接近全量微调。

**我们的场景**：1.5B 模型 + QLoRA，Kaggle T4 (16GB) 完全跑得下，单 epoch 约 2-3 分钟。

---

## 3. 全局常量 & 随机种子

In [ ]:
import os, random, numpy as np, torch, sys

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_LEN = 512

# ────────── 自动识别环境：Kaggle / Colab / 本地 ──────────
if os.path.isdir("/kaggle/input"):
    # Kaggle：数据集挂载到 /kaggle/input/lora-lesson2-data/
    DATA_DIR = "/kaggle/input/lora-lesson2-data" if os.path.isdir("/kaggle/input/lora-lesson2-data") else "./data"
    OUTPUT_DIR = "/kaggle/working/lora_qlora_out"
elif "google.colab" in sys.modules or os.path.isdir("/content"):
    # Colab：自动挂载 Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/data"
    OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/lora_qlora_out"
else:
    # 本地 / Mac
    DATA_DIR = "./data"
    OUTPUT_DIR = "./checkpoints/lora_qlora_out"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"MODEL      : {MODEL_NAME}")
print(f"DATA_DIR   : {DATA_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")


---

## 4. 加载数据集

数据集由 `data/build_dataset.py` 用 CopilotX API 合成。格式：Alpaca-style 单轮指令：

```json
{"instruction": "请解释 Incoterms 2020 中 FOB 术语", "input": "", "output": "FOB（Free On Board）..."}
```

- `train.jsonl`：500 条（学生用于训练）
- `test.jsonl`：50 条（**盲测集**，学生不能看，课堂评分用）

> 💡 如果 `data/train.jsonl` 不存在，会自动用 `seeds.jsonl`（20 条人工样本）兜底，方便快速调通流程。

In [ ]:
import json
from pathlib import Path
from datasets import Dataset

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_path = Path(DATA_DIR) / "train.jsonl"
test_path  = Path(DATA_DIR) / "test.jsonl"

if not train_path.exists():
    print(f"⚠️ {train_path} 不存在，用 seeds.jsonl 兜底（20 条）")
    train_path = Path(DATA_DIR) / "seeds.jsonl"
    test_path  = Path(DATA_DIR) / "seeds.jsonl"   # 同一份，仅供 smoke test

train_raw = load_jsonl(train_path)
test_raw  = load_jsonl(test_path)
print(f"train : {len(train_raw)} 条")
print(f"test  : {len(test_raw)} 条")
print("\n第 1 条样例：")
print(json.dumps(train_raw[0], ensure_ascii=False, indent=2))


---

## 5. 加载 Qwen2.5-1.5B-Instruct（Kaggle → 4bit QLoRA / Mac → fp16）

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if MODE == "qlora":
    # ───────────── Kaggle / CUDA 分支：4bit QLoRA ─────────────
    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",          # NF4：比 INT4 精度高
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,     # 双重量化，再省一点显存
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
else:
    # ───────────── Mac / CPU 分支：fp16 LoRA（不量化） ─────────────
    # 1.5B fp16 ≈ 3GB，M1/M2/M3/M4 的 8GB+ 统一内存够用
    dtype = torch.float16 if MODE == "fp16_lora" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=dtype,
        trust_remote_code=True,
    ).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"模型加载完成：{type(model).__name__}")
print(f"参数总量    ：{total_params:,}  (~{total_params/1e6:.0f} M)")
print(f"存储精度    ：{next(model.parameters()).dtype}")


---

## 6. 配置 LoRA（peft 三板斧之一：`LoraConfig`）

配置说明：
- **`r=8`, `lora_alpha=16`**：和第一课一致，`α/r = 2` 的缩放
- **`target_modules`**：挂到 Qwen 的 QKVO + FFN（共 7 个 proj），比第一课更激进——因为 SFT 任务需要更强的表达能力
- **`lora_dropout=0.05`**：防止小数据过拟合
- **`task_type="CAUSAL_LM"`**：声明是因果语言建模
- **`bias="none"`**：不训练 bias（实验表明对 LoRA 效果影响很小）

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# QLoRA 前必须做 "prepare_model_for_kbit_training"：
#   - 冻结所有量化层
#   - 把 LayerNorm 转 fp32（数值稳定）
#   - 关闭 use_cache（训练时会冲突）
if MODE == "qlora":
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    # Qwen2 系列：Attn 的 4 个 proj + FFN 的 3 个 proj
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# 典型输出：trainable params: 8,798,208 || all params: 1,552,743,424 || trainable%: 0.566


---

## 7. 数据处理：套 Chat Template + 只对 assistant 段计算 loss

Qwen2.5-Instruct 用的是 ChatML 格式。我们要把 Alpaca 格式转成它认识的对话格式：

```
<|im_start|>user
{instruction}<|im_end|>
<|im_start|>assistant
{output}<|im_end|>
```

**关键技巧**：loss 只在 **assistant 段**计算（prompt 段的 label 置为 -100），避免模型学「怎么提问题」而是学「怎么回答」。

In [ ]:
def format_and_tokenize(example):
    """把一条 Alpaca 样本转成 input_ids + labels（prompt 段 mask 成 -100）"""
    user_msg = example["instruction"]
    if example.get("input"):
        user_msg = f"{user_msg}\n{example['input']}"

    # 用 tokenizer 自带的 chat template 生成 prompt（不含 assistant 回答）
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_msg}],
        tokenize=False, add_generation_prompt=True,
    )
    # 完整对话（含 assistant 回答）
    full = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_msg},
         {"role": "assistant", "content": example["output"]}],
        tokenize=False, add_generation_prompt=False,
    )

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    full_ids   = tokenizer(full,   add_special_tokens=False)["input_ids"]

    # labels：prompt 段置 -100，assistant 段保留原 token
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]

    # 截断到 MAX_LEN
    input_ids = full_ids[:MAX_LEN]
    labels    = labels[:MAX_LEN]

    # 注意：不返回 attention_mask，让 DataCollatorForSeq2Seq 在 pad 时统一生成
    # （collator 会根据 input_ids 长度自动补 pad 并生成对应的 attention_mask）
    return {
        "input_ids": input_ids,
        "labels":    labels,
    }


train_ds = Dataset.from_list(train_raw).map(format_and_tokenize, remove_columns=["instruction", "input", "output"])
print(f"train: {len(train_ds)}  avg_len: {np.mean([len(x) for x in train_ds['input_ids']]):.0f}")

# 瞄一眼 labels 的 mask 情况
n_total  = len(train_ds[0]["labels"])
n_masked = sum(1 for x in train_ds[0]["labels"] if x == -100)
print(f"样本 0：总长 {n_total}，mask 掉 prompt {n_masked}，训练 loss 只看后 {n_total - n_masked} 个 token")

---

## 8. 训练（Transformers `Trainer`）

用标准 `Trainer` 就够——`peft` 模型在 API 上和普通模型完全兼容，只是 `save_pretrained` 时只会存 adapter（几 MB，而不是几 GB）。

**超参说明**：
- `num_train_epochs=3`：500 条数据，3 epoch 约 190 步，够收敛
- `per_device_train_batch_size=2 + grad_accum=4` → 有效 batch = 8
- `learning_rate=2e-4`：LoRA 常用学习率，比全量微调的 1e-5 大一个量级
- `warmup_ratio=0.05`：前 5% 步数线性热身
- `fp16=True` (CUDA) / `bf16=False` (T4 不稳)

> ⏱️ **真实训练时长**（首次跑要算上 Qwen2.5-1.5B 下载 ~3GB）：
> - **Kaggle T4** / **Colab T4**：训练本身 ≈ 10-20 分钟（加下载首次 25-30 分钟）
> - **Mac MPS**：1.5B 不实际，notebook 自动 `max_steps=2` 只验证流程（每 step 3-10 分钟）
>
> 想要**真训练 + 拿到好分数**必须用 GPU。Mac 只用来跑通流程 / 后续推理。


In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

# ────────── 按设备分支调超参 ──────────
# CUDA（Kaggle T4 / Colab T4）：3 epoch 全量训练 ── 真训练
# Mac MPS：只跑 2 step ── 流程验证（1.5B 在 MPS 上太慢，完整训练去 Colab）
# CPU：1 step 兜底
if DEVICE == "cuda":
    n_epochs       = 3
    max_steps_arg  = -1                  # -1 = 不限制，跑完 n_epochs
    use_grad_ckpt  = True
    use_fp16_arg   = True
else:
    # Mac MPS / CPU：只验证流程，不真训
    n_epochs       = 1
    max_steps_arg  = 2                   # 跑 2 个优化 step 就停
    use_grad_ckpt  = False
    use_fp16_arg   = False
    print("━" * 60)
    print(f"⚠️  当前 DEVICE={DEVICE}，1.5B 模型完整训练在此设备上不实际")
    print(f"    → 只跑 max_steps={max_steps_arg} 验证流程通")
    print(f"    → 真训练请用 Kaggle T4 / Colab T4（约 15-25 min）")
    print("━" * 60)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=n_epochs,
    max_steps=max_steps_arg,             # Mac/CPU 走少量 step
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    fp16=use_fp16_arg,
    bf16=False,
    gradient_checkpointing=use_grad_ckpt,
    report_to="none",
    remove_unused_columns=False,
    optim="paged_adamw_8bit" if MODE == "qlora" else "adamw_torch",
    seed=SEED,
)

collator = DataCollatorForSeq2Seq(tokenizer, padding=True, return_tensors="pt",
                                   label_pad_token_id=-100)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=collator,
)

print(f"\n=== 训练配置 ===")
print(f"设备            : {DEVICE} / {MODE}")
print(f"模型 device     : {next(model.parameters()).device}")
print(f"模型 dtype      : {next(model.parameters()).dtype}")
print(f"epochs / max_steps: {n_epochs} / {max_steps_arg}")
print(f"训练样本        : {len(train_ds)}")
print(f"fp16            : {use_fp16_arg}")
print(f"grad ckpt       : {use_grad_ckpt}")
print(f"================\n")

trainer.train()
print("✅ 训练完成")


---

## 9. 保存 Adapter（peft 三板斧之二：`save_pretrained`）

Adapter **只保存 LoRA 的 A, B 矩阵**——Qwen2.5-1.5B 的 adapter 大约 **30-40 MB**（对比完整模型 3 GB）。

In [ ]:
adapter_dir = os.path.join(OUTPUT_DIR, "adapter")
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

# 统计 adapter 文件大小
total_mb = sum(os.path.getsize(os.path.join(adapter_dir, f))
                for f in os.listdir(adapter_dir)) / 1e6
print(f"Adapter 已保存：{adapter_dir}")
print(f"总大小：{total_mb:.2f} MB  (对比完整 1.5B 模型 ~3000 MB，压缩 {3000/total_mb:.0f}×)")
print("\n文件清单：")
for f in sorted(os.listdir(adapter_dir)):
    size = os.path.getsize(os.path.join(adapter_dir, f)) / 1e6
    print(f"  {f:<30s} {size:>8.2f} MB")


---

## 10. 推理对比：基座 vs 微调后

用 `peft` 的 `disable_adapter` 上下文管理器，优雅地关/开 LoRA 做对比推理。

In [ ]:
model.eval()

@torch.no_grad()
def chat(prompt: str, max_new_tokens: int = 300) -> str:
    msgs = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(
        **ids, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    new_tokens = out[0, ids["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


demo_questions = [
    "请用 100 字以内解释 Incoterms 2020 中 DPU 和 DAP 的区别。",
    "HS 编码 8517.13 代表什么商品？出口需要注意什么？",
    "集装箱运输应该用 FOB 还是 FCA，为什么？",
]

for q in demo_questions:
    print(f"\n{'='*70}\n❓ {q}\n{'-'*70}")
    with model.disable_adapter():
        base_ans = chat(q)
    print(f"📦 基座:\n{base_ans}\n")
    print("-"*70)
    lora_ans = chat(q)
    print(f"✨ LoRA 后:\n{lora_ans}")


---

## 11. 盲测集评分（BLEU + ROUGE-L）

在 `test.jsonl`（50 条）上跑推理，和 reference 比对。详细实现见 [`eval.py`](./eval.py)，这里演示 notebook 内的快捷调用。

> 📌 评分是**单点分数**，不是排行榜——目的是让学生看到 LoRA 训练前后的数值差异。

In [ ]:
import jieba
from contextlib import nullcontext
from sacrebleu import corpus_bleu
from rouge_score import rouge_scorer
from tqdm.auto import tqdm

def tokenize_zh(s: str) -> str:
    """用 jieba 对中文分词后再用空格拼接——BLEU/ROUGE 需要 token 级对齐"""
    return " ".join(jieba.lcut(s))


def eval_on_test(model, test_raw, max_new_tokens=300, use_adapter=True) -> dict:
    preds, refs = [], []
    # disable_adapter() 是 context manager，整个 batch 在同一个 context 里跑
    ctx = model.disable_adapter() if not use_adapter else nullcontext()
    with ctx:
        for ex in tqdm(test_raw, desc="eval"):
            p = chat(ex["instruction"], max_new_tokens)
            preds.append(tokenize_zh(p))
            refs.append(tokenize_zh(ex["output"]))

    bleu = corpus_bleu(preds, [refs], tokenize="none").score
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    rouges = [scorer.score(r, p)["rougeL"].fmeasure for r, p in zip(refs, preds)]
    rouge_l = float(np.mean(rouges)) * 100

    return {"BLEU": bleu, "ROUGE-L": rouge_l, "n": len(preds)}


# 为节省时间，课堂上只跑前 10 条演示；作业要求跑全量 50 条
n_demo = min(10, len(test_raw))
print(f"\n--- 跑前 {n_demo} 条盲测样本 ---")
demo_subset = test_raw[:n_demo]

base_scores = eval_on_test(model, demo_subset, use_adapter=False)
lora_scores = eval_on_test(model, demo_subset, use_adapter=True)

print(f"\n{'Config':<10} | {'BLEU':>8} | {'ROUGE-L':>8}")
print("-" * 34)
print(f"{'基座':<10} | {base_scores['BLEU']:>8.2f} | {base_scores['ROUGE-L']:>8.2f}")
print(f"{'LoRA 后':<10} | {lora_scores['BLEU']:>8.2f} | {lora_scores['ROUGE-L']:>8.2f}")

---

## 12. 合并权重（peft 三板斧之三：`merge_and_unload`）

**重要**：`merge_and_unload` 只能在**非量化**模型上做——4bit 量化的底座无法直接和 fp16 的 LoRA 相加。如果要导出合并后的模型部署：

```python
# 方案 A（推荐）：重新用 fp16 加载底座，再合并
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
merged = PeftModel.from_pretrained(base, adapter_dir).merge_and_unload()
merged.save_pretrained("./merged_model")  # 完整模型，约 3 GB

# 方案 B：推理时保留 adapter，不 merge
#   优点：adapter 小，可以在同一个底座上热切换多个任务
#   缺点：每次前向多一次 B·A 矩阵乘法（开销很小）
```

下面演示方案 B 的 adapter 加载：

In [ ]:
# 演示：一个全新 kernel 里怎么加载「底座 + adapter」做推理
# 下面的代码块**展示用**，不真的 reload 当前 model（保留训练好的实例）

demo_code = f'''
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

# 1) 加载底座（QLoRA 环境下仍然用 4bit 量化底座 + 非量化 adapter）
tokenizer = AutoTokenizer.from_pretrained("{MODEL_NAME}", trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(
    "{MODEL_NAME}",
    # quantization_config=bnb_config,    # Kaggle
    torch_dtype=torch.float16,            # Mac
    device_map="auto", trust_remote_code=True,
)

# 2) 挂 adapter
model = PeftModel.from_pretrained(base, "{adapter_dir}")
model.eval()
'''
print(demo_code)
print("↑ 上面的代码就是别人用你的 adapter 的全部步骤")


---

## 🎓 今日小结

1. **peft 三板斧**：`LoraConfig` → `get_peft_model` → `save_pretrained`（再加 `merge_and_unload` 可选）
2. **QLoRA** = 4bit 量化底座 + fp16 LoRA，让 1.5B 模型在 T4 上舒服跑
3. **跨设备切换**：Kaggle 走 QLoRA，Mac 走 fp16 LoRA，一份 notebook 用 `MODE` 变量区分
4. **Adapter 只存 A/B**：30 MB 承载了 1.5B 模型的「外贸能力」，可独立分发
5. **Loss mask**：SFT 时 prompt 段置 -100，模型只学怎么**答**，不学怎么**问**
6. **BLEU + ROUGE-L**：中文评测前必须先 jieba 分词

## 📋 作业（一周内提交）

详见 [`homework.md`](./homework.md)。核心项：
- [ ] 跑通全量训练 + 50 条盲测集评分（BLEU + ROUGE-L）
- [ ] 至少做 1 个 ablation（rank ∈ {4, 8, 16} 或 target_modules 变化）
- [ ] 分析 1-2 个失败案例（200 字）
- [ ] 组间展示：5min 讲配置 + 盲测分 + 洞见

---

> 🦞 **下一步**：作业做完后，你会得到一个能答 Incoterms + HS 编码的「外贸小助手」adapter。
> 想部署？把 adapter 传到 HuggingFace Hub 或 `ollama` 本地推理就行——核心原理你已经掌握了。